![Los Angeles skyline](la_skyline.jpg)

Los Angeles, California 😎. The City of Angels. Tinseltown. The Entertainment Capital of the World! 

Known for its warm weather, palm trees, sprawling coastline, and Hollywood, along with producing some of the most iconic films and songs. However, as with any highly populated city, it isn't always glamorous and there can be a large volume of crime. That's where you can help!

You have been asked to support the Los Angeles Police Department (LAPD) by analyzing crime data to identify patterns in criminal behavior. They plan to use your insights to allocate resources effectively to tackle various crimes in different areas.

## The Data

They have provided you with a single dataset to use. A summary and preview are provided below.

It is a modified version of the original data, which is publicly available from Los Angeles Open Data.

# crimes.csv

| Column     | Description              |
|------------|--------------------------|
| `'DR_NO'` | Division of Records Number: Official file number made up of a 2-digit year, area ID, and 5 digits. |
| `'Date Rptd'` | Date reported - MM/DD/YYYY. |
| `'DATE OCC'` | Date of occurrence - MM/DD/YYYY. |
| `'TIME OCC'` | In 24-hour military time. |
| `'AREA NAME'` | The 21 Geographic Areas or Patrol Divisions are also given a name designation that references a landmark or the surrounding community that it is responsible for. For example, the 77th Street Division is located at the intersection of South Broadway and 77th Street, serving neighborhoods in South Los Angeles. |
| `'Crm Cd Desc'` | Indicates the crime committed. |
| `'Vict Age'` | Victim's age in years. |
| `'Vict Sex'` | Victim's sex: `F`: Female, `M`: Male, `X`: Unknown. |
| `'Vict Descent'` | Victim's descent:<ul><li>`A` - Other Asian</li><li>`B` - Black</li><li>`C` - Chinese</li><li>`D` - Cambodian</li><li>`F` - Filipino</li><li>`G` - Guamanian</li><li>`H` - Hispanic/Latin/Mexican</li><li>`I` - American Indian/Alaskan Native</li><li>`J` - Japanese</li><li>`K` - Korean</li><li>`L` - Laotian</li><li>`O` - Other</li><li>`P` - Pacific Islander</li><li>`S` - Samoan</li><li>`U` - Hawaiian</li><li>`V` - Vietnamese</li><li>`W` - White</li><li>`X` - Unknown</li><li>`Z` - Asian Indian</li> |
| `'Weapon Desc'` | Description of the weapon used (if applicable). |
| `'Status Desc'` | Crime status. |
| `'LOCATION'` | Street address of the crime. |

## Setup

Load the LAPD crime reports and the Python libraries used in this analysis. `TIME OCC` is read as text so values such as `0930` keep their leading zero. Re-run the next cell whenever you need a fresh copy of `crimes`.

In [17]:
# Re-run this cell to reload libraries and the crime data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read TIME OCC as text so values like "0930" keep the leading zero
crimes = pd.read_csv("crimes.csv", dtype={"TIME OCC": str})

# Preview the first few rows and confirm the dataset loaded as expected
crimes.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA NAME,Crm Cd Desc,Vict Age,Vict Sex,Vict Descent,Weapon Desc,Status Desc,LOCATION
0,220314085,2022-07-22,2020-05-12,1110,Southwest,THEFT OF IDENTITY,27,F,B,NaN,Invest Cont,2500 S SYCAMORE AV
1,222013040,2022-08-06,2020-06-04,1620,Olympic,THEFT OF IDENTITY,60,M,H,NaN,Invest Cont,3300 SAN MARINO ST
2,220614831,2022-08-18,2020-08-17,1200,Hollywood,THEFT OF IDENTITY,28,M,H,NaN,Invest Cont,1900 TRANSIENT
3,231207725,2023-02-27,2020-01-27,0635,77th Street,THEFT OF IDENTITY,37,M,H,NaN,Invest Cont,6200 4TH AV
4,220213256,2022-07-14,2020-07-14,0900,Rampart,THEFT OF IDENTITY,79,M,B,NaN,Invest Cont,1200 W 7TH ST


## 1. When do crimes peak?

`TIME OCC` is stored as 24-hour military time (for example, `1430` is 2:30 p.m.). Dividing that value by `100` converts it to an hour of the day (`0`–`23`).

The hour with the most crimes is stored in `peak_crime_hour`.

In [18]:
# Which hour has the highest frequency of crimes?
# Store the answer as an integer in peak_crime_hour.

# Military time is HHMM (e.g. "1430"). Integer-divide by 100 to get the hour (14).
crimes["HOUR"] = crimes["TIME OCC"].astype(int) // 100

# Count crimes by hour and take the hour with the largest count
peak_crime_hour = crimes["HOUR"].value_counts().idxmax()
print(f"{peak_crime_hour} hour has the highest frequency of crimes")

12 hour has the highest frequency of crimes


## 2. Where are night crimes most common?

Night crimes are those that occurred between **10:00 p.m. and 3:59 a.m.** (hours `22`–`23` and `0`–`3`).

Because that window crosses midnight, the filter uses **or**: late evening **or** the early morning hours. The patrol division with the most night crimes is stored in `peak_night_crime_location`.

In [19]:
# Which area has the most night crimes (10:00 p.m. through 3:59 a.m.)?
# Save the area name as a string in peak_night_crime_location.

# Night wraps past midnight, so keep hours 22–23 (10 p.m.–11 p.m.) or 0–3 (midnight–3:59 a.m.)
night_crimes = crimes[(crimes["HOUR"] >= 22) | (crimes["HOUR"] <= 3)]

# Count night crimes by patrol area and keep the area with the highest count
peak_night_crime_location = night_crimes["AREA NAME"].value_counts().idxmax()
print(f"{peak_night_crime_location} has the largest frequency of night crimes")

Central has the largest frequency of night crimes


## 3. How old are the victims?

Group each victim into one of seven age bands: `0-17`, `18-25`, `26-34`, `35-44`, `45-54`, `55-64`, and `65+`.

`pd.cut()` assigns ages to those bins. The resulting `victim_ages` Series uses the band labels as the index and the number of crimes as the values.

In [20]:
# Count crimes by victim age group.
# Store a Series named victim_ages: index = age labels, values = crime counts.

# Bin edges are right-inclusive by default, so 0–17, 18–25, ..., 65+
bins = [0, 17, 25, 34, 44, 54, 64, float("inf")]
labels = ["0-17", "18-25", "26-34", "35-44", "45-54", "55-64", "65+"]

# Assign each victim to an age band
crimes["AGE GROUPS"] = pd.cut(crimes["Vict Age"], bins=bins, labels=labels)

# Count crimes in each band; sort_values() ranks groups from least to most frequent
victim_ages = crimes["AGE GROUPS"].value_counts().sort_values()
print(victim_ages)

0-17      4528
65+      14747
55-64    20169
18-25    28291
45-54    28353
35-44    42157
26-34    47470
Name: AGE GROUPS, dtype: int64


## Findings

- **Peak hour:** noon (`12`) has the highest number of reported crimes.
- **Night crime hotspot:** Central has the most crimes between 10:00 p.m. and 3:59 a.m.
- **Victim ages:** adults aged 26–34 are the largest victim group, followed by 35–44. Victims aged 0–17 are the smallest group.

These patterns can help the LAPD decide when and where to add patrols, and which age groups may need targeted outreach.